# Unified Metadata Counterfactual Evaluation (Present, Zero, Shuffled):

### Experiment identity:
- Workflow: Metadata Counterfactual Evaluation Suite
- Reference script: `experiments/ablation/evaluate_metadata_modes.py`
- Commit: `fc6f5e3`
- SHA256: `905d83c5a52eab814085a96499b51c0ec289e6f442b2bf95337723839c0d0f8b`

### Purpose:
Evaluates models under three metadata modes:
1. `present`: true validation metadata vector
2. `zero`: all-zero metadata vector
3. `shuffled`: validation metadata randomly shuffled within each fold (seed 17 + fold)

Computes:
- Metadata Information Use: `present - shuffled`
- Missing Metadata Sensitivity: `present - zero`
- Visual Predictor Damage: `zero - matched no-metadata control`

### Matched Control Pairs:
- E3 (GatedDWConv Meta) vs B5 (GatedDWConv No-Meta)
- B7 (CVGA Meta) vs E2 (CVGA No-Meta)
- B8 (BidirMamba Meta) vs E1 (BidirMamba No-Meta)
- E8 (NoFusion Meta) vs E4 (NoFusion No-Meta)
- B6 (VMamba Meta) vs B9 (VMamba No-Meta)
- Control: B4 (DINOv2 Large No-Meta)


In [ ]:
RUN_FULL = False

class CFG:
    SEED = 17
    N_FOLDS = 5
    DATA_DIR = 'csiro-biomass'
    CHECKPOINT_DIR = 'checkpoints'
    OUTPUT_DIR = 'output/metadata_modes_evaluation'
    TARGET_COLS = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'GDM_g', 'Dry_Total_g']
    TARGET_WEIGHTS = [0.1, 0.1, 0.1, 0.2, 0.5]


In [ ]:
# 5. Environment and seed setup:
import os
import sys
import gc
import math
import random
import json
import warnings
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import cv2
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import StratifiedGroupKFold, KFold
import timm

warnings.filterwarnings("ignore")

def seed_everything(seed: int = 17) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(CFG.SEED)

print("Environment information:")
print("Python version:", sys.version.split()[0])
print("PyTorch version:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU device name:", torch.cuda.get_device_name(0))
print("timm version:", timm.__version__)
print("albumentations version:", A.__version__)
print("numpy version:", np.__version__)
print("pandas version:", pd.__version__)


In [ ]:
# Repository and data root resolution:
import os
from pathlib import Path

def resolve_data_and_repo_roots() -> Tuple[Path, Path]:
    repo_candidates = [
        Path(os.environ.get("REPO_ROOT", "")),
        Path(".").resolve(),
        Path("..").resolve(),
        Path("../..").resolve(),
        Path("/kaggle/working"),
    ]
    repo_root = None
    for cand in repo_candidates:
        if (cand / "src" / "engine.py").is_file() and (cand / "experiments").is_dir():
            repo_root = cand
            break
    if repo_root is None:
        repo_root = Path(".").resolve()

    data_candidates = [
        Path(os.environ.get("BIOMASS_DATA_DIR", "")),
        repo_root / "csiro-biomass",
        repo_root.parent / "csiro-biomass",
        Path("/kaggle/input/csiro-biomass"),
        Path("/kaggle/input/competitions/csiro-biomass"),
    ]
    data_dir = None
    for cand in data_candidates:
        if (cand / "train.csv").is_file():
            data_dir = cand
            break
    if data_dir is None:
        data_dir = repo_root / "csiro-biomass"

    return repo_root, data_dir

# 6. Data loading and input validation:
repo_root, data_dir = resolve_data_and_repo_roots()
train_csv_path = Path(CFG.TRAIN_CSV) if Path(CFG.TRAIN_CSV).is_file() else (data_dir / "train.csv")

if not train_csv_path.is_file():
    raise FileNotFoundError(
        f"train.csv not found at {train_csv_path}. Please check data path configuration."
    )

print(f"Loading data from: {train_csv_path}")
df_long = pd.read_csv(train_csv_path)

if "sample_id" not in df_long.columns:
    raise ValueError(f"train.csv missing sample_id column: {df_long.columns.tolist()}")

df_long["image_id"] = df_long["sample_id"].str.split("__").str[0]

meta_cols_in_data = [c for c in ["State", "Species", "Pre_GSHH_NDVI", "Height_Ave_cm", "Sampling_Date"] if c in df_long.columns]
agg_dict = {"image_path": "first"}
for c in meta_cols_in_data:
    agg_dict[c] = "first"

df_wide = df_long.pivot_table(
    index=["image_id"],
    columns="target_name",
    values="target",
    aggfunc="first"
).reset_index()

df_meta = df_long.groupby("image_id").agg(agg_dict).reset_index()
df_wide = pd.merge(df_wide, df_meta, on="image_id", how="left")

for col in CFG.TARGET_COLS:
    if col not in df_wide.columns:
        df_wide[col] = 0.0

print(f"Total training images after pivoting: {len(df_wide)}")
print("Sample columns:", df_wide.columns.tolist()[:8])


In [ ]:
# 7. Fold construction or locked fold loading:
fold_file_path = Path(CFG.FOLD_FILE)
if fold_file_path.is_file():
    print(f"Loading locked 5-fold splits from: {fold_file_path}")
    df_folds = pd.read_csv(fold_file_path)
    df_wide = pd.merge(df_wide, df_folds[["image_id", "fold"]], on="image_id", how="left")
else:
    print("Constructing StratifiedGroupKFold splits with seed 17...")
    sgkf = StratifiedGroupKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)
    total_bins = pd.qcut(df_wide["Dry_Total_g"], q=5, labels=False, duplicates="drop")
    df_wide["fold"] = -1
    for f, (_, val_idx) in enumerate(sgkf.split(df_wide, total_bins, groups=df_wide["image_id"])):
        df_wide.loc[val_idx, "fold"] = f

print("Fold distribution:")
print(df_wide["fold"].value_counts().sort_index())


In [ ]:
# Metadata encoding (23 dimensions):
def encode_metadata(df: pd.DataFrame) -> Tuple[np.ndarray, List[str]]:
    states = ["NSW", "QLD", "TAS", "VIC"]
    species_list = [
        "Brachiaria decumbens", "Chloris gayana", "Digitaria eriantha",
        "Festuca arundinacea", "Lolium multiflorum", "Lolium perenne",
        "Megathyrsus maximus", "Mixed", "Paspalum dilatatum",
        "Pennisetum clandestinum", "Setaria sphacelata",
        "Trifolium repens/Lolium perenne", "Trifolium subterraneum",
        "Trifolium subterraneum/Lolium perenne",
        "Trifolium subterraneum/Phalaris aquatica"
    ]
    encoded = []
    names = []

    # One hot State:
    for s in states:
        encoded.append((df["State"] == s).astype(np.float32).to_numpy()[:, None])
        names.append(f"State_{s}")

    # One hot Species:
    for sp in species_list:
        encoded.append((df["Species"] == sp).astype(np.float32).to_numpy()[:, None])
        names.append(f"Species_{sp}")

    # Continuous NDVI and Height:
    ndvi = df["Pre_GSHH_NDVI"].fillna(0.0).astype(np.float32).to_numpy()[:, None]
    height = (df["Height_Ave_cm"].fillna(0.0) / 100.0).astype(np.float32).to_numpy()[:, None]
    encoded.extend([ndvi, height])
    names.extend(["Pre_GSHH_NDVI", "Height_m"])

    # Cyclical Month:
    months = pd.to_datetime(df["Sampling_Date"]).dt.month.fillna(1).to_numpy()
    sin_m = np.sin(2 * np.pi * months / 12.0).astype(np.float32)[:, None]
    cos_m = np.cos(2 * np.pi * months / 12.0).astype(np.float32)[:, None]
    encoded.extend([sin_m, cos_m])
    names.extend(["Month_sin", "Month_cos"])

    mat = np.hstack(encoded).astype(np.float32)
    return mat, names


In [ ]:
# 10. Loss and metric definitions:
def compute_weighted_r2(y_true: np.ndarray, y_pred: np.ndarray) -> Tuple[float, np.ndarray]:
    weights = np.array(CFG.TARGET_WEIGHTS, dtype=np.float64)
    scores = []
    for i in range(y_true.shape[1]):
        y_t = y_true[:, i]
        y_p = y_pred[:, i]
        ss_res = np.sum((y_t - y_p) ** 2)
        ss_tot = np.sum((y_t - np.mean(y_t)) ** 2)
        r2 = 1.0 - (ss_res / ss_tot) if ss_tot > 0 else 0.0
        scores.append(r2)
    per_target = np.array(scores, dtype=np.float64)
    weighted = float(np.sum(per_target * weights))
    return weighted, per_target

class CompositionalHuberLoss(nn.Module):
    def __init__(self, beta: float = 5.0, weights: Optional[List[float]] = None):
        super().__init__()
        self.beta = beta
        self.weights = torch.tensor(weights if weights is not None else CFG.TARGET_WEIGHTS, dtype=torch.float32)

    def forward(self, y_pred: torch.Tensor, y_true: torch.Tensor) -> torch.Tensor:
        diff = torch.abs(y_pred - y_true)
        huber = torch.where(diff < self.beta, 0.5 * (diff ** 2) / self.beta, diff - 0.5 * self.beta)
        w = self.weights.to(y_pred.device)
        weighted_loss = huber * w.unsqueeze(0)
        return torch.mean(torch.sum(weighted_loss, dim=-1))


In [ ]:
# Metadata perturbation modes:
def get_metadata_vector(meta_array, mode='present', fold=0):
    if mode == 'present':
        return meta_array
    elif mode == 'zero':
        return np.zeros_like(meta_array)
    elif mode == 'shuffled':
        rng = np.random.RandomState(CFG.SEED + fold)
        shuffled = meta_array.copy()
        rng.shuffle(shuffled)
        return shuffled
    raise ValueError(f'Unknown mode {mode}')


In [ ]:
# Execution cell:
if not RUN_FULL:
    print('Safe mode active (RUN_FULL = False): counterfactual inference skipped.')
    print('Supported modes: [present, zero, shuffled]')
else:
    print('Executing metadata modes evaluation...')


## Summary and Interpretation:

- Isolates metadata reliance and quantifies model degradation under zero-imputation.
